# Генератор полусинтетического набора данных Seoul Bike Sharing Demand

Ноутбук формирует индивидуальный учебный набор данных для семестрового проекта по дисциплине **«Вероятностно-статистические основы машинного обучения»**.

Естественная основа: **Seoul Bike Sharing Demand**, UCI Machine Learning Repository.

- Официальная страница: https://archive.ics.uci.edu/dataset/560/seoul+Bike+Sharing+Demand
- DOI: https://doi.org/10.24432/C5F62R
- Лицензия: CC BY 4.0

Генератор:

1. загружает исходный CSV из UCI либо читает локальный CSV/ZIP;
2. оставляет наблюдения, в которых сервис велопроката функционировал;
3. формирует 11 столбцов студенческого набора;
4. создаёт управляемые пропуски, выбросы, группу A/B-эксперимента, экспериментальный эффект и бинарную цель;
5. сохраняет студенческий файл, расширенный преподавательский файл и JSON-паспорт варианта.

## 1. Настройка варианта

Все параметры, которые преподаватель обычно меняет, собраны в одной конфигурации.

`sample_size` задаёт число строк итогового набора **после фильтрации неработающих часов сервиса**. Значение `None` означает использование всех доступных строк.

Если `source_path=None`, исходный ZIP-архив автоматически загружается с UCI. Можно указать путь к заранее скачанному ZIP-архиву или CSV-файлу.

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

@dataclass(frozen=True)
class GenerationConfig:
    # Идентификация варианта
    variant_id: str = "01"

    # Размер и воспроизводимость выборки
    sample_size: Optional[int] = 6000
    random_seed: int = 42

    # A/B-эксперимент
    treatment_share: float = 0.50
    treatment_effect: float = 0.04

    # Пропуски во влажности
    missing_rate: float = 0.03

    # Выбросы в скорости ветра
    outlier_rate: float = 0.005
    outlier_multiplier_min: float = 2.5
    outlier_multiplier_max: float = 3.5
    wind_speed_upper_bound: float = 20.0

    # Бинарная цель высокого спроса
    high_demand_quantile: float = 0.75

    # Технические пути
    source_path: Optional[str] = None
    output_dir: str = "generated_data"


CONFIG = GenerationConfig()
CONFIG

## 2. Функции загрузки и генерации

Случайные операции используют независимые потоки, порождённые одним `random_seed`. Поэтому изменение доли пропусков не меняет распределение по экспериментальным группам, а изменение параметров выбросов не меняет выбранную подвыборку.

In [ ]:
from __future__ import annotations

import hashlib
import io
import json
import re
import unicodedata
import urllib.request
import zipfile
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


UCI_DATASET_PAGE = "https://archive.ics.uci.edu/dataset/560/seoul+Bike+Sharing+Demand"
UCI_DOWNLOAD_URL = (
    "https://archive.ics.uci.edu/static/public/560/"
    "seoul%2Bbike%2Bsharing%2Bdemand.zip"
)
UCI_DOI = "https://doi.org/10.24432/C5F62R"
LICENSE_NAME = "Creative Commons Attribution 4.0 International (CC BY 4.0)"
LICENSE_URL = "https://creativecommons.org/licenses/by/4.0/"

STUDENT_COLUMNS = [
    "date",
    "hour",
    "temperature_c",
    "humidity_pct",
    "wind_speed_m_s",
    "rain_flag",
    "season",
    "holiday",
    "experiment_group",
    "rented_bike_count",
    "high_demand",
]


def validate_config(config: GenerationConfig) -> None:
    """Проверяет допустимость параметров до начала генерации."""
    if not str(config.variant_id).strip():
        raise ValueError("variant_id не должен быть пустым.")

    if config.sample_size is not None:
        if not isinstance(config.sample_size, int) or config.sample_size <= 0:
            raise ValueError("sample_size должен быть положительным целым числом или None.")

    if not isinstance(config.random_seed, int):
        raise ValueError("random_seed должен быть целым числом.")

    bounded_rates = {
        "treatment_share": config.treatment_share,
        "missing_rate": config.missing_rate,
        "outlier_rate": config.outlier_rate,
        "high_demand_quantile": config.high_demand_quantile,
    }
    for name, value in bounded_rates.items():
        if not 0 <= value <= 1:
            raise ValueError(f"{name} должен находиться в диапазоне [0, 1].")

    if config.treatment_share in {0, 1}:
        raise ValueError("treatment_share должен создавать обе группы: значение строго между 0 и 1.")

    if config.high_demand_quantile in {0, 1}:
        raise ValueError("high_demand_quantile должен быть строго между 0 и 1.")

    if config.treatment_effect <= -1:
        raise ValueError("treatment_effect должен быть больше -1, иначе спрос станет отрицательным.")

    if config.outlier_multiplier_min <= 1:
        raise ValueError("outlier_multiplier_min должен быть больше 1.")

    if config.outlier_multiplier_max < config.outlier_multiplier_min:
        raise ValueError(
            "outlier_multiplier_max не может быть меньше outlier_multiplier_min."
        )

    if config.wind_speed_upper_bound <= 0:
        raise ValueError("wind_speed_upper_bound должен быть положительным.")


def _normalized_column_name(name: Any) -> str:
    """Приводит исходное название столбца к устойчивому ключу."""
    text = unicodedata.normalize("NFKD", str(name))
    text = text.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", "", text.casefold())


COLUMN_ALIASES = {
    "date": {"date"},
    "baseline_count": {"rentedbikecount", "rentedbikecounts"},
    "hour": {"hour"},
    "temperature_c": {"temperaturec", "temperature"},
    "humidity_pct": {"humidity", "humiditypct", "humiditypercent"},
    "wind_speed_m_s": {"windspeedms", "windspeed"},
    "rainfall_mm": {"rainfallmm", "rainfall"},
    "season": {"seasons", "season"},
    "holiday_source": {"holiday"},
    "functioning_day": {"functioningday", "functionalday"},
}


def _resolve_columns(source: pd.DataFrame) -> dict[str, str]:
    """Сопоставляет реальные названия столбцов UCI с внутренними именами."""
    normalized_to_original = {
        _normalized_column_name(column): column for column in source.columns
    }

    resolved: dict[str, str] = {}
    for canonical_name, aliases in COLUMN_ALIASES.items():
        matches = [
            normalized_to_original[alias]
            for alias in aliases
            if alias in normalized_to_original
        ]
        if not matches:
            available = ", ".join(map(str, source.columns))
            raise KeyError(
                f"Не найден обязательный столбец {canonical_name!r}. "
                f"Доступные столбцы: {available}"
            )
        resolved[canonical_name] = matches[0]

    return resolved


def _read_csv_bytes(raw_bytes: bytes) -> pd.DataFrame:
    """Читает CSV, перебирая кодировки, встречающиеся в исходном файле UCI."""
    errors: list[str] = []
    for encoding in ("utf-8-sig", "cp949", "euc-kr", "latin1"):
        try:
            frame = pd.read_csv(io.BytesIO(raw_bytes), encoding=encoding)
            if frame.shape[1] > 1:
                return frame
        except Exception as exc:
            errors.append(f"{encoding}: {exc}")

    raise ValueError(
        "Не удалось прочитать CSV ни в одной из поддерживаемых кодировок. "
        + " | ".join(errors)
    )


def _download_uci_archive(destination: Path) -> Path:
    """Загружает официальный ZIP-архив UCI в локальный кэш."""
    destination.parent.mkdir(parents=True, exist_ok=True)

    request = urllib.request.Request(
        UCI_DOWNLOAD_URL,
        headers={"User-Agent": "Mozilla/5.0 (educational dataset generator)"},
    )
    try:
        with urllib.request.urlopen(request, timeout=90) as response:
            destination.write_bytes(response.read())
    except Exception as exc:
        raise RuntimeError(
            "Не удалось автоматически загрузить архив UCI. "
            "Скачайте Seoul Bike Sharing Demand вручную и укажите путь "
            "в CONFIG.source_path."
        ) from exc

    return destination


def load_source_data(
    source_path: str | None,
    cache_dir: str | Path = "source_cache",
) -> tuple[pd.DataFrame, Path]:
    """
    Загружает исходный набор из локального CSV/ZIP или из официального архива UCI.
    Возвращает DataFrame и фактически использованный путь.
    """
    if source_path is None:
        path = Path(cache_dir) / "seoul_bike_sharing_demand.zip"
        if not path.exists():
            _download_uci_archive(path)
    else:
        path = Path(source_path).expanduser()

    if not path.exists():
        raise FileNotFoundError(f"Исходный файл не найден: {path.resolve()}")

    suffix = path.suffix.casefold()
    if suffix == ".zip":
        with zipfile.ZipFile(path) as archive:
            csv_members = [
                name for name in archive.namelist()
                if name.casefold().endswith(".csv")
            ]
            if not csv_members:
                raise FileNotFoundError("В ZIP-архиве не найден CSV-файл.")
            preferred = next(
                (
                    name for name in csv_members
                    if Path(name).name.casefold() == "seoulbikedata.csv"
                ),
                csv_members[0],
            )
            raw_bytes = archive.read(preferred)
        source = _read_csv_bytes(raw_bytes)
    elif suffix == ".csv":
        source = _read_csv_bytes(path.read_bytes())
    else:
        raise ValueError("source_path должен указывать на CSV- или ZIP-файл.")

    if source.empty:
        raise ValueError("Исходный набор данных пуст.")

    return source, path


def prepare_natural_base(source: pd.DataFrame) -> pd.DataFrame:
    """
    Оставляет необходимые естественные признаки и наблюдения,
    в которых сервис велопроката функционировал.
    """
    columns = _resolve_columns(source)

    functioning = (
        source[columns["functioning_day"]]
        .astype(str)
        .str.strip()
        .str.casefold()
    )
    functioning_mask = functioning.isin(
        {"yes", "y", "true", "1", "functional", "functioning", "func"}
    )
    if not functioning_mask.any():
        unique_values = sorted(functioning.dropna().unique().tolist())
        raise ValueError(
            "Не удалось определить работающие часы сервиса по столбцу "
            f"Functioning Day. Найденные значения: {unique_values}"
        )

    filtered = source.loc[functioning_mask].copy()

    holiday_text = (
        filtered[columns["holiday_source"]]
        .astype(str)
        .str.strip()
        .str.casefold()
    )
    holiday = np.where(holiday_text.str.contains("no", na=False), 0, 1)

    base = pd.DataFrame(
        {
            "source_row_id": filtered.index.astype(int),
            "date": pd.to_datetime(
                filtered[columns["date"]],
                dayfirst=True,
                errors="coerce",
            ),
            "hour": pd.to_numeric(filtered[columns["hour"]], errors="coerce"),
            "temperature_c": pd.to_numeric(
                filtered[columns["temperature_c"]], errors="coerce"
            ),
            "humidity_pct": pd.to_numeric(
                filtered[columns["humidity_pct"]], errors="coerce"
            ),
            "wind_speed_m_s": pd.to_numeric(
                filtered[columns["wind_speed_m_s"]], errors="coerce"
            ),
            "rainfall_mm": pd.to_numeric(
                filtered[columns["rainfall_mm"]], errors="coerce"
            ),
            "season": (
                filtered[columns["season"]]
                .astype(str)
                .str.strip()
                .str.title()
            ),
            "holiday": holiday.astype(int),
            "baseline_count": pd.to_numeric(
                filtered[columns["baseline_count"]], errors="coerce"
            ),
        }
    )

    required = [
        "date",
        "hour",
        "temperature_c",
        "humidity_pct",
        "wind_speed_m_s",
        "rainfall_mm",
        "season",
        "holiday",
        "baseline_count",
    ]
    invalid_rows = base[required].isna().any(axis=1)
    if invalid_rows.any():
        raise ValueError(
            "После преобразования исходных данных обнаружены "
            f"{int(invalid_rows.sum())} строк с некорректными обязательными значениями."
        )

    base["hour"] = base["hour"].astype(int)
    base["baseline_count"] = np.rint(base["baseline_count"]).astype(int)

    if not base["hour"].between(0, 23).all():
        raise ValueError("Исходный столбец Hour содержит значения вне диапазона 0–23.")

    if (base["baseline_count"] < 0).any():
        raise ValueError("Исходное количество аренд содержит отрицательные значения.")

    return (
        base.sort_values(["date", "hour", "source_row_id"])
        .reset_index(drop=True)
    )


def _rate_to_count(rate: float, sample_size: int) -> int:
    """Переводит долю в целое число строк; ненулевая доля даёт минимум одну строку."""
    if rate == 0:
        return 0
    return min(sample_size, max(1, int(round(rate * sample_size))))


def generate_variant(
    natural_base: pd.DataFrame,
    config: GenerationConfig,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    """
    Создаёт студенческий набор, расширенную преподавательскую версию
    и словарь метаданных.
    """
    validate_config(config)

    available_rows = len(natural_base)
    requested_size = config.sample_size
    if requested_size is None:
        sample_size = available_rows
    else:
        sample_size = requested_size

    if sample_size > available_rows:
        raise ValueError(
            f"sample_size={sample_size} превышает число доступных строк "
            f"после фильтрации: {available_rows}."
        )

    seed_sequence = np.random.SeedSequence(config.random_seed)
    sample_seed, group_seed, missing_seed, outlier_seed = seed_sequence.spawn(4)
    rng_sample = np.random.default_rng(sample_seed)
    rng_group = np.random.default_rng(group_seed)
    rng_missing = np.random.default_rng(missing_seed)
    rng_outlier = np.random.default_rng(outlier_seed)

    if sample_size == available_rows:
        selected = natural_base.copy()
    else:
        positions = rng_sample.choice(
            available_rows,
            size=sample_size,
            replace=False,
        )
        selected = natural_base.iloc[positions].copy()

    selected = (
        selected.sort_values(["date", "hour", "source_row_id"])
        .reset_index(drop=True)
    )
    n_rows = len(selected)

    high_demand_threshold = float(
        selected["baseline_count"].quantile(config.high_demand_quantile)
    )

    n_treatment = int(round(n_rows * config.treatment_share))
    treatment_positions = rng_group.choice(
        n_rows,
        size=n_treatment,
        replace=False,
    )
    treatment_mask = np.zeros(n_rows, dtype=bool)
    treatment_mask[treatment_positions] = True

    experiment_group = np.full(n_rows, "control", dtype=object)
    experiment_group[treatment_mask] = "treatment"

    rented_bike_count = selected["baseline_count"].to_numpy(dtype=float)
    rented_bike_count[treatment_mask] *= 1 + config.treatment_effect
    rented_bike_count = np.clip(
        np.rint(rented_bike_count),
        a_min=0,
        a_max=None,
    ).astype(int)

    high_demand = (
        rented_bike_count >= high_demand_threshold
    ).astype(int)

    humidity = selected["humidity_pct"].to_numpy(dtype=float).copy()
    n_missing = _rate_to_count(config.missing_rate, n_rows)
    missing_positions = (
        rng_missing.choice(n_rows, size=n_missing, replace=False)
        if n_missing
        else np.array([], dtype=int)
    )
    humidity_missing_injected = np.zeros(n_rows, dtype=bool)
    humidity_missing_injected[missing_positions] = True
    humidity[missing_positions] = np.nan

    wind = selected["wind_speed_m_s"].to_numpy(dtype=float).copy()
    n_outliers = _rate_to_count(config.outlier_rate, n_rows)

    eligible_positions = np.flatnonzero(
        (wind > 0)
        & (wind < config.wind_speed_upper_bound)
    )
    if n_outliers > len(eligible_positions):
        raise ValueError(
            "Недостаточно положительных значений скорости ветра ниже верхней "
            "границы для создания заданного числа выбросов."
        )

    outlier_positions = (
        rng_outlier.choice(
            eligible_positions,
            size=n_outliers,
            replace=False,
        )
        if n_outliers
        else np.array([], dtype=int)
    )
    multipliers = np.ones(n_rows, dtype=float)
    if n_outliers:
        multipliers[outlier_positions] = rng_outlier.uniform(
            config.outlier_multiplier_min,
            config.outlier_multiplier_max,
            size=n_outliers,
        )
        wind[outlier_positions] = np.minimum(
            wind[outlier_positions] * multipliers[outlier_positions],
            config.wind_speed_upper_bound,
        )

    wind_outlier_injected = np.zeros(n_rows, dtype=bool)
    wind_outlier_injected[outlier_positions] = True

    rain_flag = (
        selected["rainfall_mm"].to_numpy(dtype=float) > 0
    ).astype(int)

    student = pd.DataFrame(
        {
            "date": selected["date"].dt.strftime("%Y-%m-%d"),
            "hour": selected["hour"].astype(int),
            "temperature_c": selected["temperature_c"].astype(float),
            "humidity_pct": humidity,
            "wind_speed_m_s": wind,
            "rain_flag": rain_flag,
            "season": selected["season"].astype(str),
            "holiday": selected["holiday"].astype(int),
            "experiment_group": experiment_group,
            "rented_bike_count": rented_bike_count,
            "high_demand": high_demand,
        }
    )[STUDENT_COLUMNS]

    teacher = student.copy()
    teacher.insert(0, "source_row_id", selected["source_row_id"].astype(int))
    teacher["baseline_count"] = selected["baseline_count"].astype(int)
    teacher["humidity_missing_injected"] = humidity_missing_injected
    teacher["wind_outlier_injected"] = wind_outlier_injected
    teacher["wind_outlier_multiplier"] = multipliers

    metadata: dict[str, Any] = {
        "variant_id": str(config.variant_id),
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "source": {
            "dataset": "Seoul Bike Sharing Demand",
            "uci_page": UCI_DATASET_PAGE,
            "doi": UCI_DOI,
            "license": LICENSE_NAME,
            "license_url": LICENSE_URL,
        },
        "configuration": {
            key: value
            for key, value in asdict(config).items()
            if key not in {"source_path", "output_dir"}
        },
        "computed": {
            "available_rows_after_functioning_filter": int(available_rows),
            "sample_size_actual": int(n_rows),
            "high_demand_threshold": high_demand_threshold,
            "treatment_count": int(treatment_mask.sum()),
            "control_count": int((~treatment_mask).sum()),
            "treatment_share_actual": float(treatment_mask.mean()),
            "missing_count": int(n_missing),
            "missing_rate_actual": float(n_missing / n_rows),
            "outlier_count": int(n_outliers),
            "outlier_rate_actual": float(n_outliers / n_rows),
            "high_demand_count": int(high_demand.sum()),
            "high_demand_share_actual": float(high_demand.mean()),
        },
        "student_columns": STUDENT_COLUMNS,
        "teacher_only_columns": [
            "source_row_id",
            "baseline_count",
            "humidity_missing_injected",
            "wind_outlier_injected",
            "wind_outlier_multiplier",
        ],
        "rules": {
            "regression_target": "rented_bike_count",
            "exclude_from_regression_features": ["high_demand"],
            "classification_target": "high_demand",
            "exclude_from_classification_features": ["rented_bike_count"],
            "high_demand_threshold_basis": (
                "Квантиль baseline_count в выбранной выборке до воздействия."
            ),
        },
    }

    validate_generated_data(student, teacher, metadata, config)
    return student, teacher, metadata


def validate_generated_data(
    student: pd.DataFrame,
    teacher: pd.DataFrame,
    metadata: dict[str, Any],
    config: GenerationConfig,
) -> None:
    """Проверяет схему и основные инварианты сгенерированного варианта."""
    if student.columns.tolist() != STUDENT_COLUMNS:
        raise AssertionError("Нарушен утверждённый порядок столбцов студенческого файла.")

    expected_rows = (
        metadata["computed"]["sample_size_actual"]
    )
    if len(student) != expected_rows or len(teacher) != expected_rows:
        raise AssertionError("Размер сгенерированных таблиц не совпадает с паспортом.")

    allowed_missing_columns = {"humidity_pct"}
    unexpected_missing = {
        column
        for column in student.columns
        if student[column].isna().any() and column not in allowed_missing_columns
    }
    if unexpected_missing:
        raise AssertionError(
            f"Обнаружены непредусмотренные пропуски: {sorted(unexpected_missing)}"
        )

    if not student["hour"].between(0, 23).all():
        raise AssertionError("hour содержит значения вне диапазона 0–23.")

    if not student["rain_flag"].isin([0, 1]).all():
        raise AssertionError("rain_flag должен содержать только 0 и 1.")

    if not student["holiday"].isin([0, 1]).all():
        raise AssertionError("holiday должен содержать только 0 и 1.")

    if not student["high_demand"].isin([0, 1]).all():
        raise AssertionError("high_demand должен содержать только 0 и 1.")

    if set(student["experiment_group"].unique()) != {"control", "treatment"}:
        raise AssertionError("В experiment_group должны присутствовать обе группы.")

    if (student["rented_bike_count"] < 0).any():
        raise AssertionError("rented_bike_count не может быть отрицательным.")

    if (student["wind_speed_m_s"] > config.wind_speed_upper_bound + 1e-12).any():
        raise AssertionError("Скорость ветра превышает заданную верхнюю границу.")

    missing_actual = int(student["humidity_pct"].isna().sum())
    if missing_actual != metadata["computed"]["missing_count"]:
        raise AssertionError("Фактическое число пропусков не совпадает с паспортом.")

    outliers_actual = int(teacher["wind_outlier_injected"].sum())
    if outliers_actual != metadata["computed"]["outlier_count"]:
        raise AssertionError("Фактическое число выбросов не совпадает с паспортом.")

    threshold = metadata["computed"]["high_demand_threshold"]
    expected_high_demand = (
        student["rented_bike_count"].to_numpy() >= threshold
    ).astype(int)
    if not np.array_equal(
        student["high_demand"].to_numpy(),
        expected_high_demand,
    ):
        raise AssertionError("high_demand сформирован не по сохранённому порогу.")


def _json_ready_config(config: GenerationConfig) -> dict[str, Any]:
    values = asdict(config)
    return {
        key: str(value) if isinstance(value, Path) else value
        for key, value in values.items()
    }


def sha256_file(path: Path) -> str:
    """Вычисляет контрольную сумму файла."""
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_variant(
    student: pd.DataFrame,
    teacher: pd.DataFrame,
    metadata: dict[str, Any],
    config: GenerationConfig,
) -> dict[str, Path]:
    """Сохраняет студенческий CSV, преподавательский CSV, JSON и атрибуцию."""
    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    safe_variant_id = re.sub(
        r"[^a-zA-Z0-9_.-]+",
        "_",
        str(config.variant_id).strip(),
    )

    student_path = output_dir / f"student_dataset_variant_{safe_variant_id}.csv"
    teacher_path = output_dir / f"teacher_dataset_variant_{safe_variant_id}.csv"
    metadata_path = output_dir / f"teacher_metadata_variant_{safe_variant_id}.json"
    attribution_path = output_dir / "ATTRIBUTION.md"

    student.to_csv(student_path, index=False, encoding="utf-8-sig")
    teacher.to_csv(teacher_path, index=False, encoding="utf-8-sig")

    metadata_to_save = dict(metadata)
    metadata_to_save["full_configuration"] = _json_ready_config(config)
    metadata_to_save["files"] = {
        "student_csv": student_path.name,
        "teacher_csv": teacher_path.name,
    }

    metadata_path.write_text(
        json.dumps(
            metadata_to_save,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    attribution_path.write_text(
        "# Атрибуция данных\n\n"
        "Набор подготовлен на основе *Seoul Bike Sharing Demand*, "
        "UCI Machine Learning Repository, DOI: 10.24432/C5F62R, "
        "лицензия CC BY 4.0. Исходные данные преобразованы и дополнены "
        "синтетическими признаками для учебного проекта.\n\n"
        f"- Официальная страница: {UCI_DATASET_PAGE}\n"
        f"- DOI: {UCI_DOI}\n"
        f"- Лицензия: {LICENSE_URL}\n",
        encoding="utf-8",
    )

    metadata_to_save["files"]["student_sha256"] = sha256_file(student_path)
    metadata_to_save["files"]["teacher_sha256"] = sha256_file(teacher_path)
    metadata_path.write_text(
        json.dumps(
            metadata_to_save,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    return {
        "student_csv": student_path,
        "teacher_csv": teacher_path,
        "metadata_json": metadata_path,
        "attribution_md": attribution_path,
    }


def build_variant_from_config(
    config: GenerationConfig,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any], dict[str, Path]]:
    """Полный цикл: загрузка, подготовка, генерация, проверка и сохранение."""
    validate_config(config)
    source, source_used = load_source_data(config.source_path)
    natural_base = prepare_natural_base(source)
    student, teacher, metadata = generate_variant(natural_base, config)
    metadata["source"]["local_source_path"] = str(source_used.resolve())
    saved_paths = save_variant(student, teacher, metadata, config)
    return student, teacher, metadata, saved_paths

## 3. Генерация набора

Запустите следующую ячейку после настройки `CONFIG`. Будут созданы:

- CSV для студента — только 11 утверждённых столбцов;
- расширенный CSV для преподавателя;
- JSON-паспорт с конфигурацией и рассчитанным порогом;
- файл атрибуции источника и лицензии.

In [ ]:
student_data, teacher_data, metadata, saved_paths = build_variant_from_config(CONFIG)

print("Вариант успешно создан.")
print(f"Строк: {len(student_data):,}")
print(f"Столбцов в студенческом файле: {student_data.shape[1]}")
print(f"Порог high_demand: {metadata['computed']['high_demand_threshold']:.3f}")
print(f"Доля treatment: {metadata['computed']['treatment_share_actual']:.3%}")
print(f"Доля high_demand: {metadata['computed']['high_demand_share_actual']:.3%}")
print(f"Число пропусков: {metadata['computed']['missing_count']}")
print(f"Число выбросов: {metadata['computed']['outlier_count']}")
print("\nСохранённые файлы:")
for file_role, path in saved_paths.items():
    print(f"- {file_role}: {path.resolve()}")

student_data.head()

## 4. Контроль сформированного варианта

Следующие таблицы помогают преподавателю быстро проверить распределение групп, бинарной цели, пропуски и синтетические выбросы.

In [ ]:
display(
    student_data[
        ["experiment_group", "high_demand"]
    ].value_counts(dropna=False).rename("count").to_frame()
)

display(
    pd.DataFrame(
        {
            "показатель": [
                "Число строк",
                "Пропуски humidity_pct",
                "Синтетические выбросы wind_speed_m_s",
                "Среднее rented_bike_count",
                "Медиана rented_bike_count",
                "Доля high_demand",
            ],
            "значение": [
                len(student_data),
                student_data["humidity_pct"].isna().sum(),
                teacher_data["wind_outlier_injected"].sum(),
                student_data["rented_bike_count"].mean(),
                student_data["rented_bike_count"].median(),
                student_data["high_demand"].mean(),
            ],
        }
    )
)

## 5. Обязательные правила использования

- Для регрессии цель — `rented_bike_count`; `high_demand` исключается из входных признаков.
- Для классификации цель — `high_demand`; `rented_bike_count` исключается из входных признаков.
- `date` применяется для анализа временной структуры и хронологического разделения; в исходном виде в модель не включается.
- Импутация, кодирование и масштабирование настраиваются только по обучающей выборке.
- JSON-паспорт и расширенный преподавательский CSV студентам не выдаются.